# Image Classification Basics

This notebook is the missing bridge between **CNN architecture ideas** and **practical supervised vision workflows**.

We will build one complete image-classification pipeline on **Fashion-MNIST**:

- inspect the dataset and label space
- train a small CNN with a manual PyTorch loop
- track optimization diagnostics over time
- evaluate with confusion matrices and per-class metrics
- study the model's mistakes instead of stopping at one accuracy number

## Configuration

Keeping the hyperparameters in one place makes the experiment easy to rerun and modify.

In [ ]:
CONFIG = {
    # Reproducibility
    "seed": 42,  # Random seed for reproducibility

    # Data
    "train_subset_size": 10_000,  # Smaller subset for fast iteration in this notebook
    "val_subset_size": 2_000,  # Validation examples sampled from the training split
    "test_subset_size": 2_000,  # Held-out test subset for final evaluation
    "batch_size": 256,  # Images per optimization step
    "num_workers": 0,  # Keep notebook execution deterministic across platforms

    # Optimization
    "learning_rate": 1e-3,  # AdamW learning rate
    "weight_decay": 1e-4,  # Mild regularization on weights
    "max_epochs": 5,  # Enough to learn the dataset without making the notebook slow
    "early_stop_patience": 2,  # Stop if validation loss stalls

    # Model
    "num_classes": 10,  # Fashion-MNIST has 10 classes
    "dropout": 0.20,  # Dropout in the classifier head
}

CONFIG

## Imports, random seed, and device

We will use plain PyTorch here because the **training loop itself** is part of what this bridge notebook is trying to teach.

In [ ]:
from pathlib import Path
import copy
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_device() -> torch.device:
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

set_seed(CONFIG["seed"])
device = get_device()
output_dir = Path("tmp/image-classification-basics")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Using device: {device}")
print(f"Notebook artifacts directory: {output_dir.resolve()}")

## Dataset metadata and transforms

Fashion-MNIST is small enough for a notebook, but it still contains real visual ambiguity: shirts vs t-shirts, coats vs pullovers, sandals vs sneakers.

In [ ]:
CLASS_NAMES = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

DATA_MEAN = (0.2860,)
DATA_STD = (0.3530,)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(DATA_MEAN, DATA_STD),
])

dataset_root = output_dir / "data"

train_full = datasets.FashionMNIST(
    root=dataset_root,
    train=True,
    download=True,
    transform=transform,
)
test_full = datasets.FashionMNIST(
    root=dataset_root,
    train=False,
    download=True,
    transform=transform,
)

print(f"Training examples available: {len(train_full):,}")
print(f"Test examples available: {len(test_full):,}")

## A single example tells us the tensor contract

Before training anything, check the shape and scale of one transformed sample.

In [ ]:
sample_image, sample_label = train_full[0]

print(f"Image tensor shape: {tuple(sample_image.shape)}")
print(f"Image tensor dtype: {sample_image.dtype}")
print(f"Label index: {sample_label}")
print(f"Label name: {CLASS_NAMES[sample_label]}")
print(f"Pixel range after normalization: [{sample_image.min():.3f}, {sample_image.max():.3f}]")

assert sample_image.shape == (1, 28, 28)
assert 0 <= sample_label < CONFIG["num_classes"]

## Visualize a few normalized examples

Normalization helps optimization, but humans should inspect **denormalized** images.

In [ ]:
def denormalize(images: torch.Tensor) -> torch.Tensor:
    mean = torch.tensor(DATA_MEAN, dtype=images.dtype, device=images.device).view(-1, 1, 1)
    std = torch.tensor(DATA_STD, dtype=images.dtype, device=images.device).view(-1, 1, 1)
    return images * std + mean

fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for idx, ax in enumerate(axes.flat):
    image, label = train_full[idx]
    ax.imshow(denormalize(image).squeeze(0), cmap="gray")
    ax.set_title(CLASS_NAMES[label], fontsize=9)
    ax.axis("off")

plt.suptitle("Fashion-MNIST samples", fontsize=14, fontweight="bold")
plt.tight_layout()

## Create train, validation, and test subsets

We use smaller deterministic subsets so the full notebook stays interactive and still demonstrates the complete workflow.

In [ ]:
generator = torch.Generator().manual_seed(CONFIG["seed"])

train_perm = torch.randperm(len(train_full), generator=generator)
test_perm = torch.randperm(len(test_full), generator=generator)

train_indices = train_perm[: CONFIG["train_subset_size"]].tolist()
val_start = CONFIG["train_subset_size"]
val_end = val_start + CONFIG["val_subset_size"]
val_indices = train_perm[val_start:val_end].tolist()
test_indices = test_perm[: CONFIG["test_subset_size"]].tolist()

train_dataset = Subset(train_full, train_indices)
val_dataset = Subset(train_full, val_indices)
test_dataset = Subset(test_full, test_indices)

print(f"Train subset size: {len(train_dataset):,}")
print(f"Validation subset size: {len(val_dataset):,}")
print(f"Test subset size: {len(test_dataset):,}")

## DataLoaders turn examples into mini-batches

The model will see tensors in batches of shape `[batch, channels, height, width]`.

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    pin_memory=(device.type != "cpu"),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=(device.type != "cpu"),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=(device.type != "cpu"),
)

batch_images, batch_labels = next(iter(train_loader))
print(f"Batch image shape: {tuple(batch_images.shape)}")
print(f"Batch label shape: {tuple(batch_labels.shape)}")

assert batch_images.ndim == 4
assert batch_images.shape[1:] == (1, 28, 28)
assert batch_labels.ndim == 1

## Why compare a linear model to a CNN?

A **linear baseline** ignores spatial structure. A **CNN** shares weights across locations and can detect local visual patterns such as edges, hems, shoe outlines, and sleeves.

In [ ]:
class LinearBaseline(nn.Module):
    def __init__(self, num_classes: int = CONFIG["num_classes"]) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class SmallCNN(nn.Module):
    def __init__(self, num_classes: int = CONFIG["num_classes"], dropout: float = CONFIG["dropout"]) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.feature_extractor = nn.Sequential(
            self.conv1,
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            self.conv2,
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.feature_extractor(x)
        return self.classifier(features)


def count_parameters(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

## Compare the parameter budgets and output shapes

A CNN can have **more structure** without necessarily requiring an unreasonable number of parameters.

In [ ]:
baseline_model = LinearBaseline()
cnn_model = SmallCNN()

comparison = pd.DataFrame(
    [
        {"model": "Linear baseline", "parameters": count_parameters(baseline_model)},
        {"model": "Small CNN", "parameters": count_parameters(cnn_model)},
    ]
)
comparison["parameters"] = comparison["parameters"].map(lambda value: f"{value:,}")
comparison

## Check the forward pass before we start training

Shape checks catch silent mistakes early.

In [ ]:
probe_batch = batch_images[:8]
baseline_logits = baseline_model(probe_batch)
cnn_logits = cnn_model(probe_batch)

print(f"Baseline logits shape: {tuple(baseline_logits.shape)}")
print(f"CNN logits shape: {tuple(cnn_logits.shape)}")

assert baseline_logits.shape == (8, CONFIG["num_classes"])
assert cnn_logits.shape == (8, CONFIG["num_classes"])

## Choose the CNN as the bridge model

The goal here is not to win a benchmark. It is to walk through the full supervised learning loop with a model that uses spatial inductive bias.

In [ ]:
model = SmallCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1,
)

print(model)

## Training and evaluation helpers

Each epoch should answer two questions:

- how well are we fitting the training data?
- does that improvement transfer to validation data?

In [ ]:
def run_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer | None = None) -> dict:
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        if is_training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_training):
            logits = model(images)
            loss = criterion(logits, labels)

            if is_training:
                loss.backward()
                optimizer.step()

        total_loss += loss.item() * labels.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += labels.size(0)

    return {
        "loss": total_loss / total_examples,
        "acc": total_correct / total_examples,
    }


def collect_predictions(model: nn.Module, loader: DataLoader) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    model.eval()
    image_batches = []
    label_batches = []
    logit_batches = []

    with torch.no_grad():
        for images, labels in loader:
            logits = model(images.to(device))
            image_batches.append(images.cpu())
            label_batches.append(labels.cpu())
            logit_batches.append(logits.cpu())

    return torch.cat(image_batches), torch.cat(label_batches), torch.cat(logit_batches)

## Train with early stopping

We keep the best validation checkpoint in memory and stop if validation loss stops improving.

In [ ]:
history = []
best_state = copy.deepcopy(model.state_dict())
best_val_loss = float("inf")
patience_counter = 0

for epoch in range(1, CONFIG["max_epochs"] + 1):
    train_metrics = run_epoch(model, train_loader, optimizer=optimizer)
    val_metrics = run_epoch(model, val_loader, optimizer=None)
    scheduler.step(val_metrics["loss"])

    current_lr = optimizer.param_groups[0]["lr"]
    history.append(
        {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["acc"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
            "lr": current_lr,
        }
    )

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_metrics['loss']:.4f} | train_acc={train_metrics['acc']:.3f} | "
        f"val_loss={val_metrics['loss']:.4f} | val_acc={val_metrics['acc']:.3f} | "
        f"lr={current_lr:.5f}"
    )

    if val_metrics["loss"] < best_val_loss:
        best_val_loss = val_metrics["loss"]
        best_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= CONFIG["early_stop_patience"]:
            print("Early stopping triggered.")
            break

history_df = pd.DataFrame(history)
history_df

## Plot the learning curves

Loss and accuracy together help us distinguish **optimization progress** from **generalization quality**.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], marker="s", label="validation")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy")
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(history_df["epoch"], history_df["train_acc"] * 100, marker="o", label="train")
axes[1].plot(history_df["epoch"], history_df["val_acc"] * 100, marker="s", label="validation")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].grid(alpha=0.3)
axes[1].legend()

axes[2].plot(history_df["epoch"], history_df["lr"], marker="o", color="purple")
axes[2].set_title("Learning rate")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("LR")
axes[2].grid(alpha=0.3)

plt.tight_layout()

## Restore the best validation checkpoint and test once

Validation guides model selection. The **test set** should stay untouched until the very end.

In [ ]:
model.load_state_dict(best_state)

test_images, test_labels, test_logits = collect_predictions(model, test_loader)
test_probabilities = torch.softmax(test_logits, dim=1)
test_confidences, test_predictions = test_probabilities.max(dim=1)

test_accuracy = (test_predictions == test_labels).float().mean().item()
print(f"Test accuracy: {test_accuracy * 100:.2f}%")

## Build a confusion matrix

Accuracy compresses everything into one number. A confusion matrix shows **which classes the model mixes up**.

In [ ]:
cm = confusion_matrix(test_labels.numpy(), test_predictions.numpy())
cm_normalized = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(cm, cmap="Blues")
axes[0].set_title("Confusion matrix (counts)")
axes[0].set_xticks(range(len(CLASS_NAMES)))
axes[0].set_yticks(range(len(CLASS_NAMES)))
axes[0].set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
axes[0].set_yticklabels(CLASS_NAMES)
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(cm_normalized, cmap="Blues", vmin=0.0, vmax=1.0)
axes[1].set_title("Confusion matrix (row-normalized)")
axes[1].set_xticks(range(len(CLASS_NAMES)))
axes[1].set_yticks(range(len(CLASS_NAMES)))
axes[1].set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
axes[1].set_yticklabels(CLASS_NAMES)
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

for ax in axes:
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")

plt.tight_layout()

## Per-class precision, recall, and F1

This separates "the model predicts this class too often" from "the model misses this class too often."

In [ ]:
report = classification_report(
    test_labels.numpy(),
    test_predictions.numpy(),
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)

per_class_metrics = (
    pd.DataFrame(report)
    .transpose()
    .iloc[: CONFIG["num_classes"]]
    .sort_values("f1-score", ascending=False)
)
per_class_metrics[["precision", "recall", "f1-score", "support"]]

## Which categories are hardest?

Sorting by F1 score tells us where the classifier struggles most.

In [ ]:
hardest_classes = (
    per_class_metrics[["precision", "recall", "f1-score"]]
    .sort_values("f1-score")
    .round(3)
)
hardest_classes

## Extract the strongest off-diagonal confusions

Off-diagonal entries are the concrete failure modes we want to understand.

In [ ]:
confusion_pairs = []
for true_idx, true_name in enumerate(CLASS_NAMES):
    for pred_idx, pred_name in enumerate(CLASS_NAMES):
        if true_idx == pred_idx:
            continue
        confusion_pairs.append(
            {
                "true_label": true_name,
                "predicted_label": pred_name,
                "count": int(cm[true_idx, pred_idx]),
                "row_fraction": float(cm_normalized[true_idx, pred_idx]),
            }
        )

top_confusions = (
    pd.DataFrame(confusion_pairs)
    .sort_values(["count", "row_fraction"], ascending=False)
    .head(10)
)
top_confusions

## Error analysis: the most confident wrong predictions

These are useful because the model is not merely uncertain. It is **confident and wrong**, which often exposes systematic biases.

In [ ]:
mistakes = test_predictions != test_labels

error_table = pd.DataFrame(
    {
        "index": torch.arange(len(test_labels))[mistakes].numpy(),
        "true_label": [CLASS_NAMES[index] for index in test_labels[mistakes].numpy()],
        "predicted_label": [CLASS_NAMES[index] for index in test_predictions[mistakes].numpy()],
        "confidence": test_confidences[mistakes].numpy(),
    }
).sort_values("confidence", ascending=False)

error_table.head(10)

## Visualize those confident mistakes

Looking at the images often explains why the confusion matrix alone looked reasonable.

In [ ]:
top_error_indices = error_table.head(12)["index"].tolist()

fig, axes = plt.subplots(3, 4, figsize=(10, 8))
for ax, sample_index in zip(axes.flat, top_error_indices):
    image = denormalize(test_images[sample_index]).squeeze(0)
    true_label = CLASS_NAMES[test_labels[sample_index].item()]
    predicted_label = CLASS_NAMES[test_predictions[sample_index].item()]
    confidence = test_confidences[sample_index].item()

    ax.imshow(image, cmap="gray")
    ax.set_title(f"T: {true_label}\nP: {predicted_label}\n{confidence:.1%}", fontsize=9, color="darkred")
    ax.axis("off")

plt.suptitle("Most confident mistakes", fontsize=14, fontweight="bold")
plt.tight_layout()

## Also inspect the least confident correct predictions

These are "barely right" cases near the model's decision boundary.

In [ ]:
correct = test_predictions == test_labels

ambiguous_correct = pd.DataFrame(
    {
        "index": torch.arange(len(test_labels))[correct].numpy(),
        "label": [CLASS_NAMES[index] for index in test_labels[correct].numpy()],
        "confidence": test_confidences[correct].numpy(),
    }
).sort_values("confidence", ascending=True)

ambiguous_correct.head(10)

## Visualize the ambiguous correct cases

These examples show what the network finds visually borderline even when it recovers the right answer.

In [ ]:
ambiguous_indices = ambiguous_correct.head(12)["index"].tolist()

fig, axes = plt.subplots(3, 4, figsize=(10, 8))
for ax, sample_index in zip(axes.flat, ambiguous_indices):
    image = denormalize(test_images[sample_index]).squeeze(0)
    label = CLASS_NAMES[test_labels[sample_index].item()]
    confidence = test_confidences[sample_index].item()

    ax.imshow(image, cmap="gray")
    ax.set_title(f"{label}\nconfidence={confidence:.1%}", fontsize=9)
    ax.axis("off")

plt.suptitle("Lowest-confidence correct predictions", fontsize=14, fontweight="bold")
plt.tight_layout()

## Compare confidence distributions for correct vs incorrect predictions

A useful classifier should usually assign **higher confidence to correct predictions** than to incorrect ones.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
bins = np.linspace(0.0, 1.0, 21)

ax.hist(test_confidences[correct].numpy(), bins=bins, alpha=0.7, label="correct")
ax.hist(test_confidences[mistakes].numpy(), bins=bins, alpha=0.7, label="incorrect")
ax.set_title("Prediction confidence")
ax.set_xlabel("Max softmax probability")
ax.set_ylabel("Number of examples")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()

## Peek at the first-layer convolution filters

Early CNN filters often resemble simple edge or contrast detectors.

In [ ]:
first_layer_filters = model.conv1.weight.detach().cpu()
vmin = float(first_layer_filters.min())
vmax = float(first_layer_filters.max())

fig, axes = plt.subplots(4, 4, figsize=(6, 6))
for idx, ax in enumerate(axes.flat):
    ax.imshow(first_layer_filters[idx, 0], cmap="RdBu", vmin=vmin, vmax=vmax)
    ax.set_title(f"Filter {idx}", fontsize=9)
    ax.axis("off")

plt.suptitle("First convolution layer filters", fontsize=14, fontweight="bold")
plt.tight_layout()

## Summarize the bridge

The point of this notebook is the workflow, not just the final score.

In [ ]:
precision, recall, f1, _ = precision_recall_fscore_support(
    test_labels.numpy(),
    test_predictions.numpy(),
    average="macro",
    zero_division=0,
)

summary = pd.Series(
    {
        "test_accuracy": round(test_accuracy, 4),
        "macro_precision": round(float(precision), 4),
        "macro_recall": round(float(recall), 4),
        "macro_f1": round(float(f1), 4),
        "best_validation_loss": round(float(best_val_loss), 4),
        "epochs_ran": int(len(history_df)),
    }
)
summary

## Key takeaways

- An image-classification pipeline is more than a model definition. It includes **splits, batching, optimization, diagnostics, and error analysis**.
- The CNN's spatial inductive bias makes it a much better fit for images than a linear classifier that only sees flattened pixels.
- Validation curves tell us **when training helps** and **when it stops helping**.
- Confusion matrices and qualitative mistakes reveal class pairs that a single accuracy number hides.
- This is the foundation you need before moving on to more advanced vision systems such as **EfficientNet**, **transfer learning**, and **detection/segmentation**.